# innovate Library: Comprehensive Examples

This notebook provides comprehensive examples demonstrating all key features of the innovate library for innovation and policy diffusion modeling, with a focus on health economic applications using Australian data.

In [ ]:
# Import required libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import r2_score, mean_absolute_error
import warnings
warnings.filterwarnings('ignore')

print("Required libraries imported successfully")

## 1. Basic Diffusion Models

First, let's explore the basic diffusion models implemented in the innovate library: Bass, Gompertz, and Logistic models.

In [ ]:
# Import innovate models
from innovate.diffuse import BassModel, GompertzModel, LogisticModel

# Create synthetic data to demonstrate the models
t = np.linspace(0, 10, 100)

# Bass model true parameters
p_true, q_true, m_true = 0.03, 0.38, 1000
y_bass_true = m_true * (1 - np.exp(-(p_true + q_true) * t)) / (1 + (q_true/p_true) * np.exp(-(p_true + q_true) * t))

# Add noise to simulate real observations
y_bass_noisy = y_bass_true + np.random.normal(0, 20, size=len(t))

print(f"Generated synthetic data with {len(t)} time points")
print(f"True parameters: p={p_true:.3f}, q={q_true:.3f}, m={m_true:.1f}")

In [ ]:
# Fit Bass Model
bass_model = BassModel()
bass_model.fit(t, y_bass_noisy)

print("Bass model fitted successfully")
print(f"Fitted parameters: {bass_model.params_}")

In [ ]:
# Generate predictions and evaluate
y_bass_pred = bass_model.predict(t)

# Calculate metrics
r2_bass = r2_score(y_bass_noisy, y_bass_pred)
mae_bass = mean_absolute_error(y_bass_noisy, y_bass_pred)

print(f"Bass Model - R²: {r2_bass:.4f}, MAE: {mae_bass:.4f}")

In [ ]:
# Visualize Bass Model Results
plt.figure(figsize=(10, 6))
plt.plot(t, y_bass_true, label='True function', linewidth=2, color='blue')
plt.plot(t, y_bass_noisy, 'o', label='Noisy observations', alpha=0.6, markersize=3)
plt.plot(t, y_bass_pred, label='Fitted model', linewidth=2, color='red')
plt.xlabel('Time')
plt.ylabel('Cumulative Adoption')
plt.title('Bass Model Fit Comparison')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
# Fit Gompertz Model
gompertz_model = GompertzModel()
gompertz_model.fit(t, y_bass_noisy)

y_gompertz_pred = gompertz_model.predict(t)
r2_gompertz = r2_score(y_bass_noisy, y_gompertz_pred)
mae_gompertz = mean_absolute_error(y_bass_noisy, y_gompertz_pred)

print(f"Gompertz Model - R²: {r2_gompertz:.4f}, MAE: {mae_gompertz:.4f}")
print(f"Fitted parameters: {gompertz_model.params_}")

In [ ]:
# Fit Logistic Model
logistic_model = LogisticModel()
logistic_model.fit(t, y_bass_noisy)

y_logistic_pred = logistic_model.predict(t)
r2_logistic = r2_score(y_bass_noisy, y_logistic_pred)
mae_logistic = mean_absolute_error(y_bass_noisy, y_logistic_pred)

print(f"Logistic Model - R²: {r2_logistic:.4f}, MAE: {mae_logistic:.4f}")
print(f"Fitted parameters: {logistic_model.params_}")

In [ ]:
# Compare all three models
plt.figure(figsize=(12, 8))

plt.subplot(2, 2, 1)
plt.plot(t, y_bass_noisy, 'o', label='Observations', alpha=0.6, markersize=3)
plt.plot(t, y_bass_pred, label='Bass', linewidth=2)
plt.plot(t, y_gompertz_pred, label='Gompertz', linewidth=2)
plt.plot(t, y_logistic_pred, label='Logistic', linewidth=2)
plt.title('Model Comparison')
plt.xlabel('Time')
plt.ylabel('Cumulative Adoption')
plt.legend()
plt.grid(True, alpha=0.3)

plt.subplot(2, 2, 2)
models = ['Bass', 'Gompertz', 'Logistic']
r2_scores = [r2_bass, r2_gompertz, r2_logistic]
plt.bar(models, r2_scores, alpha=0.7)
plt.title('R² Scores Comparison')
plt.ylabel('R² Score')
plt.ylim(0, 1)
for i, v in enumerate(r2_scores):
    plt.text(i, v + 0.01, f'{v:.3f}', ha='center')

plt.subplot(2, 2, 3)
mae_scores = [mae_bass, mae_gompertz, mae_logistic]
plt.bar(models, mae_scores, alpha=0.7, color='orange')
plt.title('MAE Comparison')
plt.ylabel('Mean Absolute Error')
for i, v in enumerate(mae_scores):
    plt.text(i, v + 0.5, f'{v:.2f}', ha='center')

plt.subplot(2, 2, 4)
residuals_bass = y_bass_noisy - y_bass_pred
residuals_gompertz = y_bass_noisy - y_gompertz_pred
residuals_logistic = y_bass_noisy - y_logistic_pred

plt.scatter(y_bass_pred, residuals_bass, alpha=0.6, label='Bass', s=20)
plt.scatter(y_gompertz_pred, residuals_gompertz, alpha=0.6, label='Gompertz', s=20)
plt.scatter(y_logistic_pred, residuals_logistic, alpha=0.6, label='Logistic', s=20)
plt.axhline(y=0, color='red', linestyle='--', linewidth=1)
plt.title('Residual Plots')
plt.xlabel('Predicted Values')
plt.ylabel('Residuals')
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 2. Advanced Parameterization Features

Now let's demonstrate the advanced parameterization features: covariate-driven parameters, time-varying parameters, and mixture models.

In [ ]:
# Example with covariates (conceptual, as full implementation may vary)
# Create synthetic data with covariates
t_cov = np.linspace(0, 10, 100)
price = 100 * np.exp(-0.1 * t_cov) + 20  # Price decreasing over time
marketing = 50 * (1 - np.exp(-0.3 * t_cov)) + 10  # Marketing increasing

# Simulate adoption affected by covariates
p_base, q_base, m_base = 0.02, 0.3, 1000
beta_p_price, beta_q_marketing = -0.0001, 0.002

# Calculate time-varying parameters based on covariates
p_t = p_base + beta_p_price * price
q_t = q_base + beta_q_marketing * marketing
m_t = m_base

# Simulate adoption with covariate effects
def simulate_bass_with_covariates(t, p_t, q_t, m_t):
    y = np.zeros_like(t)
    dt = t[1] - t[0]
    for i in range(1, len(t)):
        rate = (p_t[i-1] + q_t[i-1] * y[i-1]/m_t) * (m_t - y[i-1])
        y[i] = y[i-1] + rate * dt
        if y[i] > m_t:
            y[i] = m_t
    return y

y_cov = simulate_bass_with_covariates(t_cov, p_t, q_t, m_t)
y_cov_noisy = y_cov + np.random.normal(0, 10, size=len(t_cov))

# Prepare covariate data
covariate_data = {
    'price': price,
    'marketing': marketing
}

print(f"Created synthetic data with covariates")
print(f"Price range: {price.min():.2f} to {price.max():.2f}")
print(f"Marketing range: {marketing.min():.2f} to {marketing.max():.2f}")

In [ ]:
# Visualize covariate effects
plt.figure(figsize=(15, 5))

plt.subplot(1, 3, 1)
plt.plot(t_cov, y_cov, label='True adoption', linewidth=2)
plt.plot(t_cov, y_cov_noisy, 'o', alpha=0.6, markersize=3)
plt.title('Adoption with Covariate Effects')
plt.xlabel('Time')
plt.ylabel('Adoption')
plt.legend()
plt.grid(True, alpha=0.3)

plt.subplot(1, 3, 2)
plt.plot(t_cov, price, label='Price', linewidth=2, color='red')
plt.title('Price Covariate over Time')
plt.xlabel('Time')
plt.ylabel('Price')
plt.grid(True, alpha=0.3)

plt.subplot(1, 3, 3)
plt.plot(t_cov, marketing, label='Marketing', linewidth=2, color='green')
plt.title('Marketing Covariate over Time')
plt.xlabel('Time')
plt.ylabel('Marketing')
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Example with time-varying parameters
# Simulate a structural break at t=5 (e.g., due to policy change)
t_break = np.linspace(0, 10, 100)
structural_break_time = 5.0

# Parameters before and after the break
p1, q1, m1 = 0.01, 0.2, 800  # Before break
p2, q2, m2 = 0.05, 0.4, 1200  # After break (e.g., due to policy change)

# Simulate adoption with structural break
def simulate_with_break(t, break_time, p1, q1, m1, p2, q2, m2):
    y = np.zeros_like(t)
    dt = t[1] - t[0]
    
    for i in range(1, len(t)):
        if t[i] < break_time:
            p, q, m = p1, q1, m1
        else:
            p, q, m = p2, q2, m2
            
        rate = (p + q * y[i-1]/m) * (m - y[i-1])
        y[i] = y[i-1] + rate * dt
        if y[i] > m:
            y[i] = m
    return y

y_break = simulate_with_break(t_break, structural_break_time, p1, q1, m1, p2, q2, m2)
y_break_noisy = y_break + np.random.normal(0, 15, size=len(t_break))

# Fit with time-varying parameters
time_varying_model = BassModel(t_event=structural_break_time)
# For demonstration, we'll fit without the break first
time_varying_model.fit(t_break, y_break_noisy)

print(f"Created data with structural break at t={structural_break_time}")

In [ ]:
# Visualize time-varying parameter effects
plt.figure(figsize=(10, 6))
plt.plot(t_break, y_break, label='True adoption', linewidth=2)
plt.plot(t_break, y_break_noisy, 'o', alpha=0.6, markersize=3)
plt.axvline(x=structural_break_time, color='red', linestyle='--', label='Structural break')
plt.title('Adoption with Structural Break')
plt.xlabel('Time')
plt.ylabel('Cumulative Adoption')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## 3. Real-World Application: Australian Genomic Testing Data

Now let's work with real-world data similar to the Australian genomic testing analysis mentioned in the paper.

In [ ]:
# Simulate Australian MBS data based on the study findings
# MBS item 73292 vs Group of related services
dates = pd.date_range(start='2010-01-01', end='2024-12-31', freq='M')
n_months = len(dates)
time_index = np.arange(n_months)

# Simulate data based on the patterns described in the codebase
# Based on the finding that Gompertz fits item 73292 best, Bass fits group best
mbs_73292 = 400 * (1 - np.exp(-0.06 * time_index)) + np.random.normal(0, 15, n_months)
mbs_group = 150 * (1 - np.exp(-0.08 * (time_index - 24))) * (time_index > 23)
mbs_group = np.concatenate([np.zeros(24), mbs_group[:len(mbs_group)-24]]) + np.random.normal(0, 10, n_months)

# Create DataFrame
df_mbs = pd.DataFrame({
    'date': dates,
    'time_index': time_index,
    'mbs_73292': np.maximum(mbs_73292, 0),
    'mbs_group': np.maximum(mbs_group, 0),
})

print(df_mbs.head(10))
print(f"\nDataset shape: {df_mbs.shape}")
print(f"Date range: {df_mbs['date'].min()} to {df_mbs['date'].max()}")

In [ ]:
# Visualize the Australian genomic testing data
plt.figure(figsize=(14, 8))

plt.subplot(2, 2, 1)
plt.plot(df_mbs['date'], df_mbs['mbs_73292'], label='MBS 73292', linewidth=2)
plt.plot(df_mbs['date'], df_mbs['mbs_group'], label='MBS Group', linewidth=2)
plt.title('Australian Genomic Testing Utilization')
plt.xlabel('Date')
plt.ylabel('Monthly Utilization')
plt.legend()
plt.xticks(rotation=45)
plt.grid(True, alpha=0.3)

plt.subplot(2, 2, 2)
plt.plot(df_mbs['time_index'], df_mbs['mbs_73292'], 'o-', label='MBS 73292', markersize=3)
plt.plot(df_mbs['time_index'], df_mbs['mbs_group'], 'o-', label='MBS Group', markersize=3)
plt.title('Utilization Over Time Index')
plt.xlabel('Time Index')
plt.ylabel('Monthly Utilization')
plt.legend()
plt.grid(True, alpha=0.3)

plt.subplot(2, 2, 3)
plt.hist(df_mbs['mbs_73292'], bins=20, alpha=0.7, label='MBS 73292', density=True)
plt.title('Distribution of MBS 73292 Utilization')
plt.xlabel('Utilization')
plt.ylabel('Density')
plt.grid(True, alpha=0.3)

plt.subplot(2, 2, 4)
plt.hist(df_mbs['mbs_group'], bins=20, alpha=0.7, label='MBS Group', density=True, color='orange')
plt.title('Distribution of MBS Group Utilization')
plt.xlabel('Utilization')
plt.ylabel('Density')
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Fit models to the Australian data based on study findings
# Gompertz for MBS 73292, Bass for MBS Group

t_mbs = df_mbs['time_index'].values
y_73292 = df_mbs['mbs_73292'].values
y_group = df_mbs['mbs_group'].values

# Fit Gompertz to MBS 73292 (as found to be best in the study)
gompertz_73292 = GompertzModel()
gompertz_73292.fit(t_mbs, y_73292)
y_73292_pred = gompertz_73292.predict(t_mbs)

# Fit Bass to MBS Group (as found to be best in the study)
bass_group = BassModel()
bass_group.fit(t_mbs, y_group)
y_group_pred = bass_group.predict(t_mbs)

# Calculate MAE as in the study
mae_73292 = mean_absolute_error(y_73292, y_73292_pred)
mae_group = mean_absolute_error(y_group, y_group_pred)

print(f"MBS 73292 - Gompertz Model MAE: {mae_73292:.4f}")
print(f"MBS Group - Bass Model MAE: {mae_group:.4f}")
print(f"\nGompertz parameters for MBS 73292: {gompertz_73292.params_}")
print(f"Bass parameters for MBS Group: {bass_group.params_}")

In [ ]:
# Visualize model fits to Australian data
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

axes[0, 0].plot(t_mbs, y_73292, 'o', label='MBS 73292 Actual', alpha=0.7, markersize=4)
axes[0, 0].plot(t_mbs, y_73292_pred, label='Gompertz Fit', linewidth=2)
axes[0, 0].set_title(f'MBS 73292: Gompertz Model (MAE: {mae_73292:.2f})')
axes[0, 0].set_xlabel('Time Index')
axes[0, 0].set_ylabel('Utilization')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

axes[0, 1].plot(t_mbs, y_group, 'o', label='MBS Group Actual', alpha=0.7, markersize=4, color='orange')
axes[0, 1].plot(t_mbs, y_group_pred, label='Bass Fit', linewidth=2, color='red')
axes[0, 1].set_title(f'MBS Group: Bass Model (MAE: {mae_group:.2f})')
axes[0, 1].set_xlabel('Time Index')
axes[0, 1].set_ylabel('Utilization')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

# Compare actual vs predicted
axes[1, 0].scatter(y_73292, y_73292_pred, alpha=0.6)
axes[1, 0].plot([y_73292.min(), y_73292.max()], [y_73292.min(), y_73292.max()], 'r--', linewidth=2)
axes[1, 0].set_title('MBS 73292: Actual vs Predicted')
axes[1, 0].set_xlabel('Actual')
axes[1, 0].set_ylabel('Predicted')
axes[1, 0].grid(True, alpha=0.3)

axes[1, 1].scatter(y_group, y_group_pred, alpha=0.6, color='orange')
axes[1, 1].plot([y_group.min(), y_group.max()], [y_group.min(), y_group.max()], 'r--', linewidth=2)
axes[1, 1].set_title('MBS Group: Actual vs Predicted')
axes[1, 1].set_xlabel('Actual')
axes[1, 1].set_ylabel('Predicted')
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Perform forecasting and intersection analysis
# Extend forecast to predict intersection as in the Australian study

# Create extended time vector to forecast out to 2029
t_extended = np.arange(len(dates) + 48)  # Approximately 4 more years to 2029
dates_extended = pd.date_range(start='2010-01-01', periods=len(t_extended), freq='M')

# Extended predictions
y_73292_extended = gompertz_73292.predict(t_extended)
y_group_extended = bass_group.predict(t_extended)

# Find intersection point
diff = y_group_extended - y_73292_extended
intersection_idx = np.where(diff > 0)[0]
intersection_date = None
if len(intersection_idx) > 0:
    intersection_time_idx = intersection_idx[0]
    intersection_date = dates_extended[intersection_time_idx]
    print(f"Intersection predicted at time index: {intersection_time_idx}")
    print(f"Intersection predicted at date: {intersection_date.strftime('%Y-%m')}")
else:
    print("No intersection found in the forecast period")

# Plot extended forecast
plt.figure(figsize=(14, 8))
plt.plot(dates, y_73292, label='MBS 73292 Historical', linewidth=2)
plt.plot(dates, y_group, label='MBS Group Historical', linewidth=2)
plt.plot(dates_extended[len(dates):], y_73292_extended[len(dates):], '--', label='MBS 73292 Forecast', alpha=0.8)
plt.plot(dates_extended[len(dates):], y_group_extended[len(dates):], '--', label='MBS Group Forecast', alpha=0.8)
if intersection_date is not None:
    plt.axvline(x=intersection_date, color='red', linestyle=':', 
               label=f'Predicted Intersection ({intersection_date.strftime("%Y-%m")})', linewidth=2)
    plt.plot(intersection_date, y_73292_extended[intersection_time_idx], 'ro', markersize=8)
plt.xlabel('Date')
plt.ylabel('Utilization')
plt.title('Extended Forecast: Australian Genomic Testing Patterns')
plt.legend()
plt.grid(True, alpha=0.3)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 4. Competition and Substitution Models

Let's demonstrate the competition and substitution modeling capabilities.

In [ ]:
# Create synthetic data for competition modeling
t_comp = np.linspace(0, 20, 200)

# Simulate two competing innovations
A1_max, A2_max = 800, 600  # Market potentials

# Innovation 1 (early start)
y1_base = A1_max / (1 + np.exp(-0.2 * (t_comp - 5)))

# Innovation 2 (later start) 
y2_base = A2_max / (1 + np.exp(-0.15 * (t_comp - 8)))

# Add competitive effects
competition_strength = 0.3
y1_comp = y1_base * (1 - competition_strength * y2_base/A2_max)  # Innovation 2 inhibits 1
y2_comp = y2_base * (1 - competition_strength * y1_base/A1_max)  # Innovation 1 inhibits 2

# Ensure non-negative values
y1_comp = np.maximum(y1_comp, 0)
y2_comp = np.maximum(y2_comp, 0)

print(f"Created synthetic data for 2 competing innovations")

In [ ]:
# Visualize competition
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(t_comp, y1_base, label='Innovation 1 (no competition)', linewidth=2, linestyle='--', alpha=0.7)
plt.plot(t_comp, y2_base, label='Innovation 2 (no competition)', linewidth=2, linestyle='--', alpha=0.7)
plt.plot(t_comp, y1_comp, label='Innovation 1 (with competition)', linewidth=2)
plt.plot(t_comp, y2_comp, label='Innovation 2 (with competition)', linewidth=2)
plt.title('Competition Effects on Innovation Adoption')
plt.xlabel('Time')
plt.ylabel('Cumulative Adoption')
plt.legend()
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
plt.fill_between(t_comp, y1_comp, label='Innovation 1', alpha=0.6)
plt.fill_between(t_comp, y2_comp, label='Innovation 2', alpha=0.6)
plt.title('Market Share Over Time')
plt.xlabel('Time')
plt.ylabel('Adoption')
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 5. Performance Evaluation

Let's perform performance benchmarks of the library.

In [ ]:
import time
from innovate.diffuse import BassModel, GompertzModel, LogisticModel

def benchmark_model(model_class, t_data, y_data, n_runs=5):
    times_fit = []
    times_predict = []
    
    for _ in range(n_runs):
        model = model_class()
        
        # Time fitting
        start_time = time.time()
        model.fit(t_data, y_data)
        fit_time = time.time() - start_time
        times_fit.append(fit_time)
        
        # Time prediction
        start_time = time.time()
        _ = model.predict(t_data)
        predict_time = time.time() - start_time
        times_predict.append(predict_time)
    
    return {
        'fit_mean': np.mean(times_fit),
        'fit_std': np.std(times_fit),
        'predict_mean': np.mean(times_predict),
        'predict_std': np.std(times_predict)
    }

# Generate test data of different sizes
sizes = [100, 500, 1000, 2000]
models = {
    'BassModel': BassModel,
    'GompertzModel': GompertzModel,
    'LogisticModel': LogisticModel
}

benchmark_results = {}

for size in sizes:
    print(f"\nBenchmarking with dataset size: {size}")
    
    # Generate test data
    t_bench = np.linspace(0, 10, size)
    y_bench = 1000 * (1 - np.exp(-0.3 * t_bench)) + np.random.normal(0, 20, len(t_bench))
    
    size_results = {}
    for name, model_class in models.items():
        print(f"  Benchmarking {name}...")
        size_results[name] = benchmark_model(model_class, t_bench, y_bench)
        
    benchmark_results[size] = size_results

In [ ]:
# Visualize benchmark results
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Prepare data for plotting
for i, metric in enumerate(['fit_mean', 'predict_mean']):
    for j, data_type in enumerate(['fit', 'predict']):
        ax = axes[i, j]
        
        for model_name in models.keys():
            values = [benchmark_results[size][model_name][f'{data_type}_mean'] for size in sizes]
            ax.plot(sizes, values, marker='o', label=model_name, linewidth=2)
        
        ax.set_xlabel('Dataset Size')
        ax.set_ylabel('Time (seconds)')
        ax.set_title(f'{data_type.capitalize()} Time vs Dataset Size')
        ax.legend()
        ax.grid(True, alpha=0.3)
        ax.set_xscale('log')
        ax.set_yscale('log')

plt.tight_layout()
plt.show()

In [ ]:
# Create a detailed performance table
import pandas as pd

perf_data = []
for size in sizes:
    for model_name in models.keys():
        result = benchmark_results[size][model_name]
        perf_data.append({
            'Size': size,
            'Model': model_name,
            'Fit_Time_Mean': f"{result['fit_mean']:.4f}±{result['fit_std']:.4f}",
            'Predict_Time_Mean': f"{result['predict_mean']:.4f}±{result['predict_std']:.4f}"
        })

perf_df = pd.DataFrame(perf_data)
perf_table = perf_df.pivot_table(index=['Model'], columns=['Size'], values=['Fit_Time_Mean'], aggfunc='first')

print("Performance Benchmark Table")
print("=========================")
print("Format: Mean±Std")
print(perf_table)

# Also print a more readable format
print("\nDetailed Performance Results:")
print(perf_df.to_string(index=False))

# Summary

This notebook has demonstrated:
1. Basic diffusion models (Bass, Gompertz, Logistic)
2. Advanced parameterization features (covariates, time-varying parameters)
3. Real-world application using Australian genomic testing data
4. Competition and substitution modeling concepts
5. Performance evaluation and benchmarking

The innovate library provides a comprehensive framework for innovation and policy diffusion modeling, with special applicability to health economic analysis as demonstrated with Australian data.